# LeetCode 1213 — Intersection of Three Sorted Arrays bằng Python

**Link đề:** https://leetcode.com/problems/intersection-of-three-sorted-arrays/
**Độ khó:** Easy · **Chủ đề:** Array, Hash Table, Binary Search, Counting

> **Bài Premium: đề bài trong notebook được viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.**

```python
class Solution:
    def arraysIntersection(self, arr1: List[int], arr2: List[int], arr3: List[int]) -> List[int]:
        ...
```

## Mục tiêu học
1. Hiểu bài **giao của nhiều mảng đã sắp xếp** và vì sao "đã sắp xếp" là manh mối lớn.
2. Nắm `in` trên list vs set, phép giao `&` của `set`, `Counter`, `bisect`.
3. Thành thạo kỹ thuật **ba con trỏ (three pointers)**: luôn tiến con trỏ đang chỉ giá trị **nhỏ nhất**.
4. So sánh 5 cách: brute force, set, đếm, binary search, ba con trỏ — kèm độ phức tạp.
5. Trace từng bước, chứng minh, test đối chiếu, bài tập TODO có đáp án cuối notebook.

## 1. Đề bài và ví dụ

Cho ba mảng số nguyên `arr1`, `arr2`, `arr3` đều được **sắp xếp tăng dần nghiêm ngặt** (strictly increasing — không có phần tử trùng trong cùng một mảng).
Trả về **mảng đã sắp xếp** gồm các số nguyên xuất hiện trong **cả ba** mảng.

| `arr1` | `arr2` | `arr3` | Kết quả |
|---|---|---|---|
| `[1,2,3,4,5]` | `[1,2,5,7,9]` | `[1,3,4,5,8]` | `[1,5]` |
| `[197,418,523,876,1356]` | `[501,880,1593,1710,1870]` | `[521,682,1337,1395,1764]` | `[]` |

**Constraints (theo hiểu biết):** `1 <= arr1.length, arr2.length, arr3.length <= 1000`; `1 <= arr1[i], arr2[i], arr3[i] <= 2000`.

Giải thích ví dụ 1: `1` có trong cả ba; `2` không có trong `arr3`; `3`, `4` không có trong `arr2`; `5` có trong cả ba.

In [1]:
arr1, arr2, arr3 = [1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]
for x in arr1:
    print(x, "| trong arr2:", x in arr2, "| trong arr3:", x in arr3)

1 | trong arr2: True | trong arr3: True
2 | trong arr2: True | trong arr3: False
3 | trong arr2: False | trong arr3: True
4 | trong arr2: False | trong arr3: True
5 | trong arr2: True | trong arr3: True


## 2. Nền tảng Python cần biết

### 2.1. `x in list` — tìm tuyến tính `O(n)`

`in` trên list phải dò từ đầu tới khi gặp (hoặc hết). Với list dài, mỗi lần kiểm tra tốn `O(n)`.

### 2.2. `set` và phép giao `&`

`set` tra cứu trung bình `O(1)`. Phép `a & b` (hoặc `a.intersection(b)`) cho các phần tử chung. **Set không giữ thứ tự** → cần `sorted` khi trả kết quả.

In [2]:
s1, s2, s3 = {1, 2, 3, 4, 5}, {1, 2, 5, 7, 9}, {1, 3, 4, 5, 8}
print("s1 & s2      =", s1 & s2)
print("s1 & s2 & s3 =", s1 & s2 & s3)
print("sorted(...)  =", sorted(s1 & s2 & s3))
print("intersection:", s1.intersection(s2, s3))

s1 & s2      = {1, 2, 5}
s1 & s2 & s3 = {1, 5}
sorted(...)  = [1, 5]
intersection: {1, 5}


### 2.3. Ghép list bằng `+` và `Counter`

In [3]:
from collections import Counter
merged = [1, 2, 3] + [2, 3] + [3]
print("merged =", merged)
print(Counter(merged))

merged = [1, 2, 3, 2, 3, 3]
Counter({3: 3, 2: 2, 1: 1})


### 2.4. `bisect_left` — tìm nhị phân trên list đã sắp xếp

`bisect_left(a, x)` trả **vị trí đầu tiên** `i` mà `a[i] >= x` (vị trí chèn `x` để giữ thứ tự), trong `O(log n)`.
`x` có trong `a` ⇔ `i < len(a) and a[i] == x`.

In [4]:
from bisect import bisect_left
a = [1, 3, 4, 5, 8]
for x in [1, 2, 5, 9]:
    i = bisect_left(a, x)
    found = i < len(a) and a[i] == x
    print(f"x={x}: bisect_left -> {i}, có trong a? {found}")

x=1: bisect_left -> 0, có trong a? True
x=2: bisect_left -> 1, có trong a? False
x=5: bisect_left -> 3, có trong a? True
x=9: bisect_left -> 5, có trong a? False


### 2.5. Vòng `while` với nhiều điều kiện `and`

`while i < n1 and j < n2 and k < n3:` dừng ngay khi **một** mảng bất kỳ đã duyệt hết — vì không thể còn phần tử chung nào nữa.

In [5]:
i, j = 0, 0
a, b2 = [1, 2], [7, 8, 9]
steps = 0
while i < len(a) and j < len(b2):
    steps += 1
    i += 1
print("dừng sau", steps, "bước vì i =", i, "== len(a)")

dừng sau 2 bước vì i = 2 == len(a)


## 3. Cách 1 — Brute force: duyệt `arr1`, dùng `in` trên list

Ý nghĩ tự nhiên: với mỗi `x` trong `arr1`, kiểm tra `x in arr2 and x in arr3`. Vì duyệt `arr1` tăng dần, kết quả tự động tăng dần.

In [6]:
def intersect_brute(arr1, arr2, arr3):
    return [x for x in arr1 if x in arr2 and x in arr3]

print(intersect_brute([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]))
print(intersect_brute([197, 418, 523, 876, 1356], [501, 880, 1593, 1710, 1870], [521, 682, 1337, 1395, 1764]))

[1, 5]
[]


**Chi phí:** mỗi `in` là `O(n)` → tổng `O(n1·(n2 + n3))`. Với 1000 phần tử mỗi mảng: ~2 triệu phép so sánh — vẫn chạy, nhưng **không tận dụng** việc mảng đã sắp xếp.

## 4. Nút thắt và insight

Ta có 2 hướng cải tiến:

1. **Tăng tốc phép tra cứu** — dùng `set` (`O(1)` trung bình) hoặc `bisect` (`O(log n)`, tận dụng sắp xếp).
2. **Tận dụng triệt để thứ tự** — giống cách trộn (merge) các mảng đã sắp xếp: đặt 3 con trỏ ở đầu 3 mảng.
   - Nếu 3 giá trị bằng nhau → đó là phần tử chung, ghi nhận và **tiến cả ba**.
   - Nếu không → giá trị **nhỏ nhất** trong ba **chắc chắn không phải phần tử chung** (hai mảng kia chỉ còn các giá trị ≥ giá trị hiện tại của chúng, lớn hơn nó) → tiến con trỏ của nó.

Mỗi bước tiến ít nhất một con trỏ → tối đa `n1 + n2 + n3` bước, bộ nhớ `O(1)`.

In [7]:
a1, a2, a3 = [1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]
i = j = k = 1      # giả sử đang ở bước thứ 2: a1[1]=2, a2[1]=2, a3[1]=3
vals = (a1[i], a2[j], a3[k])
print("ba giá trị:", vals, "| nhỏ nhất:", min(vals))
print("2 có thể chung? a3 từ vị trí k chỉ còn", a3[k:], "-> không còn số 2 -> tiến i và j")

ba giá trị: (2, 2, 3) | nhỏ nhất: 2
2 có thể chung? a3 từ vị trí k chỉ còn [3, 4, 5, 8] -> không còn số 2 -> tiến i và j


## 5. Lời giải chính — giảng từng đoạn rồi ghép

### Cách 2 — Set intersection

In [8]:
def intersect_set(arr1, arr2, arr3):
    return sorted(set(arr1) & set(arr2) & set(arr3))

print(intersect_set([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]))

[1, 5]


### Cách 3 — Đếm: phần tử chung xuất hiện đúng 3 lần

Vì **mỗi mảng không có phần tử trùng**, một số xuất hiện 3 lần trong `arr1 + arr2 + arr3` ⇔ nó có mặt ở cả ba mảng. Giá trị ≤ 2000 → có thể đếm bằng mảng 2001 ô, rồi duyệt giá trị tăng dần để có kết quả đã sắp xếp mà **không cần sort**.

In [9]:
def intersect_count(arr1, arr2, arr3):
    cnt = [0] * 2001
    for arr in (arr1, arr2, arr3):
        for x in arr:
            cnt[x] += 1
    return [v for v in range(2001) if cnt[v] == 3]

print(intersect_count([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]))

[1, 5]


### Cách 4 — Binary search: duyệt mảng ngắn nhất, tìm nhị phân trong hai mảng kia

In [10]:
from bisect import bisect_left

def contains(a, x):
    i = bisect_left(a, x)
    return i < len(a) and a[i] == x

def intersect_bisect(arr1, arr2, arr3):
    a, b_, c = sorted([arr1, arr2, arr3], key=len)   # a là mảng ngắn nhất
    return [x for x in a if contains(b_, x) and contains(c, x)]

print(intersect_bisect([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]))

[1, 5]


`sorted([...], key=len)` sắp xếp **các list** theo độ dài — tham số `key` cho biết "so sánh theo cái gì".

### Cách 5 — Ba con trỏ (lời giải chính)

**Đoạn 1 — khởi tạo:** `i = j = k = 0`, `result = []`.
**Đoạn 2 — điều kiện lặp:** cả ba con trỏ còn trong mảng.
**Đoạn 3 — bằng nhau:** ghi nhận, tiến cả ba.
**Đoạn 4 — không bằng nhau:** tìm `m = min(...)`, tiến **mọi** con trỏ có giá trị bằng `m`.

In [11]:
def intersect_pointers(arr1, arr2, arr3):
    i = j = k = 0                                         # đoạn 1
    result = []
    while i < len(arr1) and j < len(arr2) and k < len(arr3):   # đoạn 2
        x, y, z = arr1[i], arr2[j], arr3[k]
        if x == y == z:                                   # đoạn 3
            result.append(x)
            i += 1; j += 1; k += 1
        else:                                             # đoạn 4
            m = min(x, y, z)
            if x == m: i += 1
            if y == m: j += 1
            if z == m: k += 1
    return result

print(intersect_pointers([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]))
print(intersect_pointers([197, 418, 523, 876, 1356], [501, 880, 1593, 1710, 1870], [521, 682, 1337, 1395, 1764]))

[1, 5]
[]


`x == y == z` là **so sánh chuỗi** trong Python: tương đương `x == y and y == z`.

Ở đoạn 4 dùng ba lệnh `if` **độc lập** (không phải `elif`): nếu hai con trỏ cùng giữ giá trị nhỏ nhất thì cả hai đều tiến — vẫn đúng, và nhanh hơn.

## 6. TRACE từng bước (ba con trỏ)

`arr1 = [1,2,3,4,5]`, `arr2 = [1,2,5,7,9]`, `arr3 = [1,3,4,5,8]`

| Bước | i,j,k | x,y,z | min | Quyết định | result |
|:---:|:---:|:---:|:---:|---|---|
| 1 | 0,0,0 | 1,1,1 | 1 | bằng nhau → lấy 1, tiến cả ba | [1] |
| 2 | 1,1,1 | 2,2,3 | 2 | tiến i, j | [1] |
| 3 | 2,2,1 | 3,5,3 | 3 | tiến i, k | [1] |
| 4 | 3,2,2 | 4,5,4 | 4 | tiến i, k | [1] |
| 5 | 4,2,3 | 5,5,5 | 5 | bằng nhau → lấy 5 | [1,5] |
| — | 5,3,4 | — | — | i = 5 = len(arr1) → dừng | [1,5] |

In [12]:
def trace_pointers(arr1, arr2, arr3):
    i = j = k = 0
    result = []
    step = 1
    while i < len(arr1) and j < len(arr2) and k < len(arr3):
        x, y, z = arr1[i], arr2[j], arr3[k]
        if x == y == z:
            result.append(x)
            print(f"bước {step}: (i,j,k)=({i},{j},{k}) giá trị=({x},{y},{z}) -> bằng nhau, lấy {x}; result={result}")
            i += 1; j += 1; k += 1
        else:
            m = min(x, y, z)
            moved = [name for name, v in (("i", x), ("j", y), ("k", z)) if v == m]
            print(f"bước {step}: (i,j,k)=({i},{j},{k}) giá trị=({x},{y},{z}) -> min={m}, tiến {moved}")
            if x == m: i += 1
            if y == m: j += 1
            if z == m: k += 1
        step += 1
    print(f"dừng: (i,j,k)=({i},{j},{k}), lens=({len(arr1)},{len(arr2)},{len(arr3)}) -> result={result}")
    return result

assert trace_pointers([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]) == [1, 5]

bước 1: (i,j,k)=(0,0,0) giá trị=(1,1,1) -> bằng nhau, lấy 1; result=[1]
bước 2: (i,j,k)=(1,1,1) giá trị=(2,2,3) -> min=2, tiến ['i', 'j']
bước 3: (i,j,k)=(2,2,1) giá trị=(3,5,3) -> min=3, tiến ['i', 'k']
bước 4: (i,j,k)=(3,2,2) giá trị=(4,5,4) -> min=4, tiến ['i', 'k']
bước 5: (i,j,k)=(4,2,3) giá trị=(5,5,5) -> bằng nhau, lấy 5; result=[1, 5]
dừng: (i,j,k)=(5,3,4), lens=(5,5,5) -> result=[1, 5]


Trace ví dụ 2 (không có phần tử chung) — xem các con trỏ "đuổi nhau" và dừng khi một mảng cạn:

In [13]:
assert trace_pointers([197, 418, 523, 876, 1356], [501, 880, 1593, 1710, 1870], [521, 682, 1337, 1395, 1764]) == []

bước 1: (i,j,k)=(0,0,0) giá trị=(197,501,521) -> min=197, tiến ['i']
bước 2: (i,j,k)=(1,0,0) giá trị=(418,501,521) -> min=418, tiến ['i']
bước 3: (i,j,k)=(2,0,0) giá trị=(523,501,521) -> min=501, tiến ['j']
bước 4: (i,j,k)=(2,1,0) giá trị=(523,880,521) -> min=521, tiến ['k']
bước 5: (i,j,k)=(2,1,1) giá trị=(523,880,682) -> min=523, tiến ['i']
bước 6: (i,j,k)=(3,1,1) giá trị=(876,880,682) -> min=682, tiến ['k']
bước 7: (i,j,k)=(3,1,2) giá trị=(876,880,1337) -> min=876, tiến ['i']
bước 8: (i,j,k)=(4,1,2) giá trị=(1356,880,1337) -> min=880, tiến ['j']
bước 9: (i,j,k)=(4,2,2) giá trị=(1356,1593,1337) -> min=1337, tiến ['k']
bước 10: (i,j,k)=(4,2,3) giá trị=(1356,1593,1395) -> min=1356, tiến ['i']
dừng: (i,j,k)=(5,2,3), lens=(5,5,5) -> result=[]


## 7. Code nộp LeetCode

### 7.1. Bản chính — ba con trỏ

In [14]:
from typing import List

class Solution:
    def arraysIntersection(self, arr1: List[int], arr2: List[int], arr3: List[int]) -> List[int]:
        i = j = k = 0
        result = []
        while i < len(arr1) and j < len(arr2) and k < len(arr3):
            x, y, z = arr1[i], arr2[j], arr3[k]
            if x == y == z:
                result.append(x)
                i += 1
                j += 1
                k += 1
            else:
                m = min(x, y, z)
                if x == m:
                    i += 1
                if y == m:
                    j += 1
                if z == m:
                    k += 1
        return result

sol = Solution()
print(sol.arraysIntersection([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]))

[1, 5]


### 7.2. Bản ngắn — set

In [15]:
class SolutionSet:
    def arraysIntersection(self, arr1: List[int], arr2: List[int], arr3: List[int]) -> List[int]:
        return sorted(set(arr1) & set(arr2) & set(arr3))

assert SolutionSet().arraysIntersection([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]) == [1, 5]
print("SolutionSet OK")

SolutionSet OK


### 7.3. Bản đếm — giữ thứ tự nhờ duyệt `arr1`

Thay vì duyệt cả miền 1..2000, duyệt lại `arr1` (đã tăng dần) và giữ các số có đếm = 3:

In [16]:
from collections import Counter

class SolutionCount:
    def arraysIntersection(self, arr1: List[int], arr2: List[int], arr3: List[int]) -> List[int]:
        cnt = Counter(arr1 + arr2 + arr3)
        return [x for x in arr1 if cnt[x] == 3]

assert SolutionCount().arraysIntersection([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]) == [1, 5]
print("SolutionCount OK")

SolutionCount OK


## 8. Vì sao ba con trỏ đúng?

**Bất biến:** mọi phần tử chung **chưa có** trong `result` đều nằm trong cả ba "phần đuôi" `arr1[i:]`, `arr2[j:]`, `arr3[k:]`; và `result` tăng dần.

- **Ban đầu** `i = j = k = 0`: phần đuôi là cả mảng → bất biến đúng.
- **Nhánh bằng nhau:** `x` có ở cả ba → thêm vào `result`. Mỗi mảng tăng nghiêm ngặt nên `x` chỉ có ở đúng vị trí hiện tại; tiến cả ba chỉ bỏ đi `x` (đã ghi nhận) → bất biến giữ.
- **Nhánh khác nhau:** gọi `m = min(x, y, z)`, giả sử `x == m` và có một giá trị khác, chẳng hạn `y > m`. Vì `arr2` tăng dần, mọi phần tử của `arr2[j:]` đều ≥ `y > m` → `m` **không** nằm trong phần đuôi của `arr2` → `m` không phải phần tử chung "còn thiếu". Bỏ `m` (tiến `i`) không làm mất gì → bất biến giữ.
- **Dừng:** khi một phần đuôi rỗng, theo bất biến không còn phần tử chung nào bị thiếu → `result` đầy đủ. Mỗi vòng tiến ≥ 1 con trỏ → dừng sau ≤ `n1 + n2 + n3` vòng.

`result` tăng dần vì các giá trị được thêm lấy từ `arr1` theo chỉ số `i` tăng dần. ∎

## 9. Độ phức tạp và so sánh

Gọi `n1, n2, n3` là độ dài, `N = n1 + n2 + n3`, `V = 2000` là giá trị tối đa.

| Cách | Time | Space phụ | Tận dụng "đã sắp xếp"? |
|---|---|---|---|
| Brute force (`in` trên list) | `O(n1·(n2 + n3))` | `O(1)` | không |
| Set intersection | `O(N + r log r)` trung bình (r = số phần tử chung) | `O(N)` | không (phải sort lại) |
| Đếm (mảng / Counter) | `O(N)` hoặc `O(N + V)` | `O(V)` hoặc `O(N)` | một phần |
| Binary search | `O(n_min · log n_max)` | `O(1)` | có |
| **Ba con trỏ** | `O(N)` | `O(1)` | **có, triệt để** |

Ba con trỏ là lời giải "đúng tinh thần" bài: tuyến tính, không bộ nhớ phụ. Binary search có lợi khi một mảng **rất ngắn** so với hai mảng kia.

## 10. Bộ test

Mẫu đề, mảng 1 phần tử, ba mảng giống hệt, không giao nhau, phần tử chung ở đầu/cuối, độ dài lệch nhau, và đối chiếu mọi cách trên dữ liệu ngẫu nhiên.

In [17]:
impls = {
    "brute": intersect_brute,
    "set": intersect_set,
    "count": intersect_count,
    "bisect": intersect_bisect,
    "pointers": intersect_pointers,
    "Solution": Solution().arraysIntersection,
    "SolSet": SolutionSet().arraysIntersection,
    "SolCount": SolutionCount().arraysIntersection,
}
cases = [
    (([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]), [1, 5]),
    (([197, 418, 523, 876, 1356], [501, 880, 1593, 1710, 1870], [521, 682, 1337, 1395, 1764]), []),
    (([1], [1], [1]), [1]),
    (([1], [2], [1]), []),
    (([1, 2, 3], [1, 2, 3], [1, 2, 3]), [1, 2, 3]),
    (([2000], [1, 2000], [5, 6, 2000]), [2000]),                # chung ở cuối
    (([1, 2, 3, 4, 5, 6, 7, 8], [4], [2, 4, 8]), [4]),          # độ dài lệch
    (([1, 3, 5], [2, 4, 6], [1, 2, 3, 4, 5, 6]), []),
]
for (a1, a2, a3), expected in cases:
    for name, f in impls.items():
        assert f(a1, a2, a3) == expected, (name, a1, a2, a3)
    print(f"{str((a1, a2, a3))[:60]:62} -> {expected} PASS")

([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8])            -> [1, 5] PASS
([197, 418, 523, 876, 1356], [501, 880, 1593, 1710, 1870], [   -> [] PASS
([1], [1], [1])                                                -> [1] PASS
([1], [2], [1])                                                -> [] PASS
([1, 2, 3], [1, 2, 3], [1, 2, 3])                              -> [1, 2, 3] PASS
([2000], [1, 2000], [5, 6, 2000])                              -> [2000] PASS
([1, 2, 3, 4, 5, 6, 7, 8], [4], [2, 4, 8])                     -> [4] PASS
([1, 3, 5], [2, 4, 6], [1, 2, 3, 4, 5, 6])                     -> [] PASS


In [18]:
import random

rng = random.Random(1213)
for _ in range(400):
    arrs = [sorted(rng.sample(range(1, 60), rng.randint(1, 30))) for _ in range(3)]
    res = {name: f(*arrs) for name, f in impls.items()}
    assert all(r == res["brute"] for r in res.values()), (arrs, res)
big = [sorted(rng.sample(range(1, 2001), 1000)) for _ in range(3)]
r = Solution().arraysIntersection(*big)
assert r == intersect_set(*big)
print("400 test ngẫu nhiên khớp nhau | mảng lớn: số phần tử chung =", len(r))

400 test ngẫu nhiên khớp nhau | mảng lớn: số phần tử chung = 239


`f(*arrs)` — dấu `*` "rải" list `arrs` thành 3 đối số riêng: `f(arrs[0], arrs[1], arrs[2])`.

## 11. Lỗi người mới hay gặp

### Lỗi 1: Trả kết quả từ `set` mà quên `sorted`

Set không đảm bảo thứ tự. Với số nhỏ, CPython có thể **tình cờ** in ra tăng dần, khiến bạn tưởng đúng — nhưng không có gì bảo đảm.

In [19]:
s = {2000, 1, 1000, 64, 33}
print("list(set) :", list(s), "(thứ tự không được bảo đảm)")
print("sorted(set):", sorted(s))

list(set) : [64, 1, 33, 2000, 1000] (thứ tự không được bảo đảm)
sorted(set): [1, 33, 64, 1000, 2000]


### Lỗi 2: Chỉ tiến **một** con trỏ khi ba giá trị bằng nhau

Nếu chỉ `i += 1`, vòng sau `arr2[j]`, `arr3[k]` vẫn là `x` cũ — không lỗi nhưng thừa bước; tệ hơn, nếu viết điều kiện kiểu khác có thể **thêm trùng**. Đúng: tiến cả ba.

### Lỗi 3: Tiến con trỏ của giá trị **lớn nhất**

In [20]:
def wrong_advance_max(arr1, arr2, arr3):
    i = j = k = 0
    res = []
    while i < len(arr1) and j < len(arr2) and k < len(arr3):
        x, y, z = arr1[i], arr2[j], arr3[k]
        if x == y == z:
            res.append(x); i += 1; j += 1; k += 1
        else:
            M = max(x, y, z)          # SAI: bỏ đi giá trị lớn nhất
            if x == M: i += 1
            elif y == M: j += 1
            else: k += 1
    return res

print("sai :", wrong_advance_max([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]))
print("đúng:", intersect_pointers([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]))

sai : [1]
đúng: [1, 5]


Vì sao sai: giá trị lớn nhất **vẫn có thể** xuất hiện ở hai mảng kia phía sau (ví dụ `3` ở bước 2 có trong `arr1`). Chỉ giá trị **nhỏ nhất** mới chắc chắn bị loại.

### Lỗi 4: Dùng điều kiện `or` thay vì `and` trong `while`

`while i < n1 or j < n2 or k < n3` → khi một mảng đã cạn vẫn đọc `arr[i]` → `IndexError`.

In [21]:
def wrong_or(arr1, arr2, arr3):
    i = j = k = 0
    res = []
    while i < len(arr1) or j < len(arr2) or k < len(arr3):
        x, y, z = arr1[i], arr2[j], arr3[k]
        if x == y == z:
            res.append(x); i += 1; j += 1; k += 1
        else:
            m = min(x, y, z)
            if x == m: i += 1
            if y == m: j += 1
            if z == m: k += 1
    return res

try:
    wrong_or([1, 2], [1, 5], [1, 9])
except IndexError as e:
    print("IndexError:", e)

IndexError: list index out of range


### Lỗi 5: Dùng cách đếm khi mảng **có thể có trùng**

Cách "đếm = 3" chỉ đúng vì mỗi mảng tăng **nghiêm ngặt**. Nếu `arr1 = [2, 2, 2]`, `arr2 = arr3 = [5]`, số `2` đếm được 3 lần dù không có ở `arr2`.

In [22]:
from collections import Counter
cnt = Counter([2, 2, 2] + [5] + [5])
print("cnt[2] =", cnt[2], "-> cách đếm sẽ nhận nhầm 2 là phần tử chung")

cnt[2] = 3 -> cách đếm sẽ nhận nhầm 2 là phần tử chung


## 12. Mẹo nhận dạng pattern

- **"Các mảng đã sắp xếp"** + **"phần tử chung / trộn / so khớp"** → **nhiều con trỏ**, tiến con trỏ có giá trị nhỏ nhất.
- Một mảng rất ngắn, mảng kia rất dài → duyệt mảng ngắn + **binary search** mảng dài.
- Không quan tâm thứ tự, cần code nhanh → `set` và `&`.
- Miền giá trị nhỏ → mảng đếm.

Bài cùng họ: 349 Intersection of Two Arrays, 350 Intersection of Two Arrays II (có trùng → đếm `min`), 88 Merge Sorted Array, 1002 Find Common Characters, 2248 Intersection of Multiple Arrays.

## 13. Biến thể

### 13.1. Giao của **k** mảng đã sắp xếp (tổng quát hoá con trỏ)

In [23]:
def intersect_k(arrays: list[list[int]]) -> list[int]:
    ptr = [0] * len(arrays)
    result = []
    while all(ptr[t] < len(arrays[t]) for t in range(len(arrays))):
        vals = [arrays[t][ptr[t]] for t in range(len(arrays))]
        m = min(vals)
        if all(v == m for v in vals):
            result.append(m)
            ptr = [p + 1 for p in ptr]
        else:
            for t in range(len(arrays)):
                if vals[t] == m:
                    ptr[t] += 1
    return result

print(intersect_k([[1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8], [0, 5, 10]]))

[5]


### 13.2. Intersection of Two Arrays II (LeetCode 350) — mảng chưa sắp xếp, **có trùng**

Mỗi phần tử xuất hiện trong kết quả **bằng số lần nhỏ nhất** giữa hai mảng → `Counter` và phép `&` của Counter (lấy min từng key).

In [24]:
from collections import Counter

def intersect_with_dups(nums1, nums2):
    common = Counter(nums1) & Counter(nums2)      # min của từng key
    return sorted(common.elements())              # elements() lặp mỗi key đúng số lần

print(intersect_with_dups([1, 2, 2, 1], [2, 2]), intersect_with_dups([4, 9, 5], [9, 4, 9, 8, 4]))

[2, 2] [4, 9]


### 13.3. Hợp (union) của ba mảng đã sắp xếp, không trùng

Cùng khung ba con trỏ: mỗi bước lấy `m = min`, thêm `m` một lần, tiến **mọi** con trỏ đang bằng `m`; khi một mảng cạn thì coi giá trị của nó là `+∞`.

In [25]:
def union_three(a1, a2, a3):
    INF = float("inf")
    i = j = k = 0
    res = []
    while i < len(a1) or j < len(a2) or k < len(a3):
        x = a1[i] if i < len(a1) else INF
        y = a2[j] if j < len(a2) else INF
        z = a3[k] if k < len(a3) else INF
        m = min(x, y, z)
        res.append(m)
        if x == m: i += 1
        if y == m: j += 1
        if z == m: k += 1
    return res

print(union_three([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]))

[1, 2, 3, 4, 5, 7, 8, 9]


Biểu thức `a if điều_kiện else b` là **toán tử ba ngôi** (conditional expression). Ở đây dùng `+∞` như "lính canh" để mảng đã cạn không bao giờ là min.

## 14. Bài tập (TODO — không gọi khi chạy)

**Bài 1 (tính tay).** `arr1=[2,4,6,8]`, `arr2=[1,2,4,8,16]`, `arr3=[4,8,12]`. Lập bảng trace ba con trỏ và cho kết quả.

**Bài 2 (dự đoán output).** `{3, 1} & {1, 2}`, `bisect_left([1, 3, 5], 4)`, `1 == 1 == 2`.

**Bài 3 (điền TODO).** Hoàn thiện ba con trỏ.

In [26]:
def exercise_intersect(arr1, arr2, arr3):
    i = j = k = 0
    result = []
    while i < len(arr1) and j < len(arr2) and k < len(arr3):
        # TODO 1: nếu ba giá trị bằng nhau -> thêm, tiến cả ba
        # TODO 2: ngược lại -> tiến các con trỏ có giá trị nhỏ nhất
        pass
    return result

# assert exercise_intersect([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]) == [1, 5]

**Bài 4 (viết lại).** Giao của **hai** mảng đã sắp xếp nghiêm ngặt bằng hai con trỏ (không dùng set).

In [27]:
def exercise_intersect_two(a, b):
    # TODO
    pass

# assert exercise_intersect_two([1, 3, 4, 6], [2, 3, 6, 7]) == [3, 6]

**Bài 5 (biến thể).** Ba mảng **đã sắp xếp nhưng có thể có trùng** (không nghiêm ngặt). Trả các giá trị chung, **mỗi giá trị một lần**.

In [28]:
def exercise_intersect_dups(arr1, arr2, arr3):
    # TODO
    pass

# assert exercise_intersect_dups([1, 1, 2, 2, 3], [1, 2, 2, 2], [1, 1, 2]) == [1, 2]

**Bài 6 (mini challenge).** Smallest Common Element in Rows (LeetCode 1198): cho ma trận `mat` mỗi hàng tăng nghiêm ngặt, trả **phần tử nhỏ nhất chung cho mọi hàng**, không có thì `-1`.

In [29]:
def exercise_smallest_common(mat: list[list[int]]) -> int:
    # TODO
    pass

# assert exercise_smallest_common([[1, 2, 3, 4, 5], [2, 4, 5, 8, 10], [3, 5, 7, 9, 11], [1, 3, 5, 7, 9]]) == 5
# assert exercise_smallest_common([[1, 2, 3], [2, 3, 4], [2, 3, 5]]) == 2
# assert exercise_smallest_common([[1, 2], [3, 4]]) == -1

## 15. Gợi ý 3 mức

| Bài | Mức 1 | Mức 2 | Mức 3 |
|---|---|---|---|
| 1 | Bắt đầu (2,1,4) | min là ai? | tiến con trỏ giữ giá trị nhỏ nhất, lặp lại |
| 2 | `&` là giao | bisect_left trả vị trí chèn | so sánh chuỗi = `and` |
| 3 | Bằng nhau hay không | `x == y == z` | `m = min(x, y, z)`; ba `if` độc lập |
| 4 | Hai con trỏ như merge | `while i < len(a) and j < len(b)` | `<` thì `i += 1`, `>` thì `j += 1`, `==` thì thêm và tiến cả hai |
| 5 | Có trùng → có thể thêm lặp | chỉ thêm nếu `result` rỗng hoặc khác phần tử cuối | `if not result or result[-1] != x: result.append(x)` |
| 6 | Mỗi hàng là một "mảng" | dùng `Counter` qua mọi hàng (mỗi hàng không trùng) | duyệt hàng 0 tăng dần, trả số đầu tiên có đếm = số hàng |

## 16. Đáp án tham khảo

**Bài 1:** kết quả `[4, 8]`. Các bước: (2,1,4)→tiến j; (2,2,4)→tiến i,j; (4,4,4)→lấy 4; (6,8,8)→tiến i; (8,8,8)→lấy 8; i cạn → dừng.

**Bài 2:** `{1}`, `2`, `False`.

In [30]:
from collections import Counter

def answer_intersect(arr1, arr2, arr3):
    i = j = k = 0
    result = []
    while i < len(arr1) and j < len(arr2) and k < len(arr3):
        x, y, z = arr1[i], arr2[j], arr3[k]
        if x == y == z:
            result.append(x); i += 1; j += 1; k += 1
        else:
            m = min(x, y, z)
            if x == m: i += 1
            if y == m: j += 1
            if z == m: k += 1
    return result

def answer_intersect_two(a, b):
    i = j = 0
    res = []
    while i < len(a) and j < len(b):
        if a[i] == b[j]:
            res.append(a[i]); i += 1; j += 1
        elif a[i] < b[j]:
            i += 1
        else:
            j += 1
    return res

def answer_intersect_dups(arr1, arr2, arr3):
    i = j = k = 0
    result = []
    while i < len(arr1) and j < len(arr2) and k < len(arr3):
        x, y, z = arr1[i], arr2[j], arr3[k]
        if x == y == z:
            if not result or result[-1] != x:
                result.append(x)
            i += 1; j += 1; k += 1
        else:
            m = min(x, y, z)
            if x == m: i += 1
            if y == m: j += 1
            if z == m: k += 1
    return result

def answer_smallest_common(mat):
    cnt = Counter(v for row in mat for v in row)
    for v in mat[0]:
        if cnt[v] == len(mat):
            return v
    return -1

assert answer_intersect([2, 4, 6, 8], [1, 2, 4, 8, 16], [4, 8, 12]) == [4, 8]
assert ({3, 1} & {1, 2}, bisect_left([1, 3, 5], 4), 1 == 1 == 2) == ({1}, 2, False)
assert answer_intersect([1, 2, 3, 4, 5], [1, 2, 5, 7, 9], [1, 3, 4, 5, 8]) == [1, 5]
assert answer_intersect_two([1, 3, 4, 6], [2, 3, 6, 7]) == [3, 6]
assert answer_intersect_dups([1, 1, 2, 2, 3], [1, 2, 2, 2], [1, 1, 2]) == [1, 2]
assert answer_smallest_common([[1, 2, 3, 4, 5], [2, 4, 5, 8, 10], [3, 5, 7, 9, 11], [1, 3, 5, 7, 9]]) == 5
assert answer_smallest_common([[1, 2, 3], [2, 3, 4], [2, 3, 5]]) == 2
assert answer_smallest_common([[1, 2], [3, 4]]) == -1
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 17. Template nhớ nhanh và checklist submit

```python
i = j = k = 0; res = []
while i < len(A) and j < len(B) and k < len(C):
    x, y, z = A[i], B[j], C[k]
    if x == y == z:
        res.append(x); i += 1; j += 1; k += 1
    else:
        m = min(x, y, z)                 # nhỏ nhất chắc chắn không chung
        if x == m: i += 1
        if y == m: j += 1
        if z == m: k += 1
```

**Câu thần chú:** *"Đã sắp xếp → con trỏ. Không bằng nhau → bỏ thằng nhỏ nhất."*

### Checklist trước khi submit
- [ ] Điều kiện `while` dùng `and` (dừng khi **một** mảng cạn)?
- [ ] Tiến con trỏ của giá trị **nhỏ nhất**, không phải lớn nhất?
- [ ] Khi bằng nhau, tiến **cả ba** con trỏ?
- [ ] Nếu dùng `set`: đã `sorted` kết quả?
- [ ] Nếu dùng cách đếm: chắc chắn mỗi mảng **không có trùng**?
- [ ] Tên method `arraysIntersection(self, arr1, arr2, arr3)`?